In [0]:
# Type casting is not necessary here as we already have to correct data types, as per our previous result of the sql query in the bronze layer code

In [0]:
#Now let's perform some data anonymization tasks, data validation tasks
from pyspark.sql import functions as F
bronze_orders_df=spark.read.table("db_retail_analytics.supply_chain_project.bronze_orders")

silver_orders_df=(
    bronze_orders_df
    #Masking the cusomer_id, this is already done according to the original dataset, but we still do again to get the concept of ananomizaiton
    .withColumn("customer_id",F.sha2(F.col("customer_id"),256))

    # checking the data quality by flagging the bad columns(data integrity)
    .withColumn(
        "is_valid_timeline",
        F.when(
            (F.col("order_delivered_customer_date").isNotNull()) & 
            (F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp")), 
            False
        ).otherwise(True)
    )

)
# saving this silver tables
(
    silver_orders_df.
    write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_orders")
)

print("The silver table is saved/overwritten")

In [0]:
%sql select * from db_retail_analytics.supply_chain_project.bronze_order_items LIMIT 5;

In [0]:
# silver layer transformations for order_items
order_items_bronze_df=(
    spark.
    read.
    table("db_retail_analytics.supply_chain_project.bronze_order_items")
)

silver_order_items_df=(
    order_items_bronze_df
    #making sure our primary key row doesn't have null in it
    .filter(F.col("order_id").isNotNull())
    #data hygeine 
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    # we need to cast but as from the above sql result we can see we don't need to cast anything because all the column we need are in the same datatype
    #data validation: keeping only the rows with price>0 as they can make bad values
    .filter(F.col("price")>0)
)
# saving our table in the Silver layer
(
    silver_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_order_items")
)

print("silver order items table save successfully")

In [0]:
'''
From this cell the code is done after the Minimum Viable Pipeline (initial vertical slice of the architecture(2 kpis created in powerbi))
'''

In [0]:
'''
As per our metrics we can understand 90% of the orders are on time, but we are going to see where the 10% of the orders are delayed
'''


In [0]:
%%sql
SELECT * FROM db_retail_analytics.supply_chain_project.bronze_geolocation LIMIT 5;

In [0]:
'''
rather than just dropping the zip_code_prefix, let's do the aggregation because for powerbi if we have the average of lat, long the map will be more interesting with locaitons pointing to the middle of the zipcode
'''
from pyspark.sql import functions as F
bronze_geo=spark.read.table("db_retail_analytics.supply_chain_project.bronze_geolocation")

silver_geo=(
    bronze_geo
    .filter(F.col("geolocation_zip_code_prefix").isNotNull())
    .groupBy(F.col("geolocation_zip_code_prefix"))
    .agg(
        F.avg(F.col("geolocation_lat")).alias("latitude"),
        F.avg(F.col("geolocation_lng")).alias("longitude"),
        #now let's remove the trailing and ending space in the text columns
        F.first(F.trim(F.col("geolocation_city"))).alias("city"),
        F.first(F.trim(F.col("geolocation_state"))).alias("state")
    )
)

#it's time we have to save this into a table

(
    silver_geo
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_geolocaiton")
)

In [0]:
'''
Now we are done with the bronze table of customers we will clean so the briging would be better to the gold tables
'''


In [0]:
from pyspark.sql import functions as F
customers_bronze_df=(
    spark.
    read.
    table("db_retail_analytics.supply_chain_project.bronze_customers")
)

silver_customers_df=(
    customers_bronze_df
    .filter(F.col("customer_id").isNotNull())
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_unique_id",F.trim(F.col("customer_unique_id")))
    .withColumn("customer_city", F.trim(F.col("customer_city")))
    .withColumn("customer_state",F.trim(F.col("customer_state")))
    .dropDuplicates(["customer_id"])
    #as we did earlier hasing the customer ID(anonymization) we are going to do the same thing here, but don't forget to use the same algorithm we used earlier, because customer ID using same alogithm as earlier gives same values as we got earlier, if not we loose the data
    .withColumn("customer_id",F.sha2(F.col("customer_id"),256))
    # The reason for select statement is to create an alias for the column names, and for the zip_code I don't change the name because i wanted to explicitly show how I join/broadcast with different names in the gold table 
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        F.col("customer_city").alias("city"),
        F.col("customer_state").alias("state")
    )
)

(
    silver_customers_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_customers")
)

In [0]:
%%sql
SELECT * FROM db_retail_analytics.supply_chain_project.bronze_customers LIMIT 5;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType

In [0]:
bronze_sellers=spark.read.table("db_retail_analytics.supply_chain_project.bronze_sellers")
bronze_prodcuts=spark.read.table("db_retail_analytics.supply_chain_project.bronze_products")


In [0]:
'''
Transformations for the sellers table include:
- checking column id is string type
- trimming around in all columns
- dropping any seller id duplicates
- saving all in a silver table
'''

In [0]:
silver_sellers=(
    bronze_sellers
    .withColumn("seller_id",F.col("seller_id").cast(StringType()))
    .withColumn("seller_zip_code_prefix",F.trim(F.col("seller_zip_code_prefix")))
    .withColumn("seller_city",F.trim(F.col("seller_city")))
    .withColumn("seller_state",F.trim(F.col("seller_state")))
    .dropDuplicates(["seller_id"])
)
(
    silver_sellers
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_sellers")
)

In [0]:
%%sql
DESCRIBE db_retail_analytics.supply_chain_project.bronze_products

In [0]:
'''
According to the above conformation from bronze_products table all the data_types are perfect
- now trim the spaces around string columns(product_id,Product_category_name)
- and let's use coalesce for the product_category_name as unknown
- dropping duplicates from prodcut_id
'''

In [0]:
silver_products=(
    bronze_prodcuts
    .withColumn("product_id",F.trim(F.col("product_id")))
    .withColumn("product_category_name",F.coalesce(F.trim(F.col("product_category_name")),F.lit("unknown")))
    .dropDuplicates(["product_id"])
)

(
    silver_products
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_products")
)

In [0]:
%%sql
DESCRIBE db_retail_analytics.supply_chain_project.bronze_reviews

In [0]:
'''
-After examining the above result looks like we need to change the datatypes of almost every column as Databricks didn't infer properly
- trimming
-dropping duplicates(review_id)
'''

In [0]:
%%sql
SELECT * FROM db_retail_analytics.supply_chain_project.bronze_reviews LIMIT 5

In [0]:
bronze_reviews=spark.read.table("db_retail_analytics.supply_chain_project.bronze_reviews")
silver_reviews=(
    bronze_reviews
    #After running once i got the error, value '2018-04-01 00:27:51' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed, So, I verified the distinct values present in the reviews column and found out the column was too messy with dates, strings, comments etc..
    #so let's use a try_cast to tolerate malformed input and return NULL instead.
    
    .withColumn("review_score", F.expr("try_cast(trim(review_score) as double)"))
    
    # Strip spaces from text columns
    .withColumn("review_comment_title", F.trim(F.col("review_comment_title")))
    .withColumn("review_comment_message", F.trim(F.col("review_comment_message")))
    
    # Safely casting messy dates to timestamp (return NULL for bad values)
    .withColumn("review_creation_date", F.expr("try_cast(trim(review_creation_date) as timestamp)"))
    .withColumn("review_answer_timestamp", F.expr("try_cast(trim(review_answer_timestamp) as timestamp)"))
    
    .dropDuplicates(["review_id"])
)

(
    silver_reviews
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_reviews")
)

In [0]:
%%sql
SELECT DISTINCT review_score FROM db_retail_analytics.supply_chain_project.bronze_reviews

In [0]:
%sql
SELECT 
    COUNT(*) as total_rows,
    COUNT(CASE WHEN review_creation_date IS NULL THEN 1 END) as broken_dates,
    COUNT(CASE WHEN review_score IS NULL THEN 1 END) as broken_scores
FROM db_retail_analytics.supply_chain_project.silver_reviews;


In [0]:
'''
from the above cell result you can see how many broken rows we have got
'''

In [0]:
%%sql
DESCRIBE db_retail_analytics.supply_chain_project.bronze_payments

In [0]:
%%sql
SELECT * FROM db_retail_analytics.supply_chain_project.bronze_payments LIMIT 5

In [0]:
'''
Transformations for the payments would need:
- datatypes are already perfect so we don't need to touch them
- for strings lets do the trimming
- I would still love to put the debit card payments because it might be useful for our cluster bar chart to show the difference if needed
- for integer columns there are no rules yet because there can be any number of payments 

'''

In [0]:
%%sql
SELECT DISTINCT(payment_type) FROM db_retail_analytics.supply_chain_project.bronze_payments

In [0]:
from pyspark.sql import functions as F

bronze_payments=spark.read.table("db_retail_analytics.supply_chain_project.bronze_payments")
silver_payments=(
    bronze_payments
    .withColumn("order_id",F.trim(F.col("order_id")))
    .withColumn("payment_type",F.trim(F.col("payment_type")))
    #we have some bad payment methods, so exclude those rows
    .filter(~F.col("payment_type").isin(["not_defined"]))
    .dropna()
)
(
    silver_payments
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("db_retail_analytics.supply_chain_project.silver_payments")

)